In [3]:
from qiskit import QuantumCircuit
from qiskit.quantum_info import Operator 
from qiskit import QuantumCircuit, transpile
#from qiskit_ibm_runtime import EstimatorV2 as Estimator
from qiskit_ibm_runtime import SamplerV2 as Sampler
from qiskit.quantum_info import PauliList
from qiskit_aer import AerSimulator
from qiskit.quantum_info import SparsePauliOp
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
import itertools
from itertools import product
import random
import numpy as np


#Total Shots
shots= 200

######################################################################
# Single qubit shadow snap-shot
#######################################################################
def single_qubit_shadow(pauli, bit):
    b = int(bit)
    proj = np.array([[1, 0], [0, 0]]) if b == 0 else np.array([[0, 0], [0, 1]])

    if pauli == 'X':
        U = np.array([[1, 1], [1, -1]]) / np.sqrt(2)
    elif pauli == 'Y':
        U = np.array([[1, -1j], [1, 1j]]) / np.sqrt(2)
    else:  # 'Z'
        U = np.eye(2)
    rho_shadow = 3 * (U.conj().T @ proj @ U) - np.eye(2)

    return rho_shadow

#####################################################################
# Two qubit shadow shap shot or tensorproduct of rho_1 * rho_2
######################################################################
def k_qubit_shadow(paulis, bitstring):
    rho = 1
    for P, b in zip(paulis, bitstring):
        rho = np.kron(rho, single_qubit_shadow(P, b))
    return rho

# -----------------------------------------------------------
# Renyi-2 entropy from independent shadow snapshots
# -----------------------------------------------------------
def renyi2_entropy_from_shadows(shadow_snapshots):

    N = len(shadow_snapshots)

    if N < 2:
        raise ValueError("At least 2 snapshots are required.")

    rho_sum = np.sum(shadow_snapshots, axis=0)

    tr_sum_sq = np.trace(rho_sum @ rho_sum).real

    diagonal_sum = sum(
        np.trace(rho @ rho).real
        for rho in shadow_snapshots
    )

    purity = (
        tr_sum_sq - diagonal_sum
    ) / (N * (N - 1))

    if purity <= 0:
        return np.nan, purity

    SA = -np.log(purity)

    return SA, purity

# -----------------------------------------------------------
# Renyi-2 entropy from independent shadow snapshots
# -----------------------------------------------------------
def renyi2_entropy_from_shadows_V2(shadow_snapshots):
    N = len(shadow_snapshots)
    purity = 0.0

    for i in range(N):
        for j in range(i + 1, N):
            purity += np.trace(shadow_snapshots[i] @ shadow_snapshots[j]).real

    purity *= 2 / (N * (N - 1))
    SA= -np.log(purity)
    return SA, purity

In [4]:

import numpy as np
import ast
import re
from pathlib import Path
from collections import Counter

def classical_shadow_density_matrix(shadow_data, k):
    dim = 2**k
    rho_sum = np.zeros((dim, dim), dtype=complex)
    for rho in shadow_data:
        rho_sum += rho
    return rho_sum / len(shadow_data)

def calculate_Renyi2(rho):

    print(f"trace rho = {np.trace(rho)}")

    # Eigenvalues
    vals = np.linalg.eigvalsh(rho)
    vals = np.real(vals)
    # -------------------------------------------
    # Moments
    # -------------------------------------------
    R1 = np.sum(vals)
    R2 = np.sum(vals**2)
    # R2 = np.real(np.trace(rho @ rho))
    
    S_A = -np.log(R2)

    print(f"R1 = {R1}")
    print(f"R2 = {R2}")

    return S_A, R1, R2
    
# -----------------------------------------------------------
# Read measurement data for one time step
# -----------------------------------------------------------
def read_shadow_data(filename):

    shadow_data = {}

    pattern = re.compile(
        r"^\s*bitstring of\s+([IXYZ ]+?)\s+(\[.*\])\s*$"
    )

    with open(filename, "r") as f:
        for line in f:

            match = pattern.match(line)

            if match is None:
                continue

            pauli_label = match.group(1).replace(" ", "")
            bitstrings = ast.literal_eval(match.group(2))

            shadow_data.setdefault(pauli_label, []).extend(
                bitstrings
            )

    return shadow_data



# Example: t = 0.0, step = 0
shadow_data_0_step = read_shadow_data(
    "bitstrings_step_0.txt"
)

shadow_counts_0 = {}

for pauli_label, bitstrings in shadow_data_0_step.items():
    shadow_counts_0[pauli_label] = dict(Counter(bitstrings))
#print(shadow_counts_0)

n_bit = 3
rho_sum = np.zeros((2**n_bit, 2**n_bit), dtype=complex)
total_shots = 0
#################################################################
shadow_snapshots = []
for pauli_label, bitstrings in shadow_data_0_step.items():

    for bit in bitstrings:
        rho_v2 =  k_qubit_shadow(pauli_label, bit)

        shadow_snapshots.append(rho_v2)

N = len(shadow_snapshots)
print("number of shadow=",N)

S_A, purity_v2 = renyi2_entropy_from_shadows(shadow_snapshots)
##########################################################
for pauli_label, counts in shadow_counts_0.items():

    for bit, count in counts.items():

        rho_pauli = k_qubit_shadow(pauli_label, bit)

        rho_sum += count * rho_pauli
        total_shots += count

rho = rho_sum / total_shots

print("Trace of rho =", np.trace(rho))

purity = np.trace(rho @ rho).real

SA = -np.log(purity) if purity > 0 else np.nan

print("Purity =", purity, purity_v2)
print("Renyi-2 entropy =", SA, S_A)

number of shadow= 5400
Trace of rho = (0.9999999999999989+0j)
Purity = 1.0318907407407403 1.0089294313761807
Renyi-2 entropy = -0.031392790079349676 -0.008889799754336264


In [5]:
shadow_data_steps = []

for step in range(11):
    filename = f"bitstrings_step_{step}.txt"

    shadow_data = read_shadow_data(filename)

    shadow_data_steps.append(shadow_data)

# -----------------------------------------------------------
# Calculate Renyi-2 entropy for each time step
# -----------------------------------------------------------
SA_list = []
purity_list = []

for i, shadow_sets in enumerate(shadow_data_steps):
    shadow_snapshots = []
    
    for pauli_label, bitstrings in shadow_sets.items():

        for bit in bitstrings:
            rho_v2 =  k_qubit_shadow(pauli_label, bit)

            shadow_snapshots.append(rho_v2)

    N = len(shadow_snapshots)
    print("number of shadow=",N)

    S_A, purity_v2 = renyi2_entropy_from_shadows(shadow_snapshots)

    SA_list.append(S_A)
    purity_list.append(purity_v2)

    print(f"Step {i}: Purity = {purity_v2:.6f}")
    print(f"Step {i}: Renyi-2 entropy = {S_A:.6f}")
    print("------------------------------------")


# Convert results into NumPy arrays
SA_array = np.array(SA_list)
purity_array = np.array(purity_list)
    

number of shadow= 5400
Step 0: Purity = 1.008929
Step 0: Renyi-2 entropy = -0.008890
------------------------------------
number of shadow= 5400
Step 1: Purity = 0.629704
Step 1: Renyi-2 entropy = 0.462505
------------------------------------
number of shadow= 5400
Step 2: Purity = 0.275565
Step 2: Renyi-2 entropy = 1.288932
------------------------------------
number of shadow= 5400
Step 3: Purity = 0.303428
Step 3: Renyi-2 entropy = 1.192610
------------------------------------
number of shadow= 5400
Step 4: Purity = 0.253714
Step 4: Renyi-2 entropy = 1.371547
------------------------------------
number of shadow= 5400
Step 5: Purity = 0.274332
Step 5: Renyi-2 entropy = 1.293417
------------------------------------
number of shadow= 5400
Step 6: Purity = 0.221782
Step 6: Renyi-2 entropy = 1.506061
------------------------------------
number of shadow= 5400
Step 7: Purity = 0.205912
Step 7: Renyi-2 entropy = 1.580307
------------------------------------
number of shadow= 5400
Step 8: 

In [6]:
np.savetxt(f"entropy_shadow_quimb_witout_AQC.txt", SA_list)